# HW10: Обучение с подкреплением / Reinforcement learning

**Wine • CPU • RU/EN**

178 реальных химических анализов вина; 13 измерений, метка одного из трёх классов сортов. Измерения считаем доступными до решения; стоимость измерения не моделируется. Нет разбиения по производителю/времени: перенос на новые партии не проверен. / 178 real wine analyses; 13 chemical measurements and three cultivar classes. Measurements are assumed available before acting; measurement costs are not modelled. No producer/time split: transfer to new batches is not assessed.

Источник / Source: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html

Реальные наблюдения; учебное правило награды 1[action==label]. Это контекстный бандит: действия не меняют будущие контексты. Метки хранятся в среде, learner получает только выбранную награду. Не промышленный журнал взаимодействий. / Real observations with a teaching reward rule 1[action==label]. This is a contextual bandit: actions do not affect future contexts. The environment stores labels; the learner receives only selected-action reward. This is not an operational interaction log.

Запустите в Colab CPU сверху вниз. Четыре задания; замените только ячейки решений, проверки сохраните. После выполнения перезапустите ядро и Run all. / Run top to bottom in Colab CPU. Four exercises; replace solution cells while preserving checks. Restart the kernel and Run all before submission.

## Начните здесь: что мы делаем? / Start here: what are we doing?

**RU.** Представьте программу, которая видит химический анализ вина и выбирает один из трёх классов: 0, 1 или 2. После выбора ей сообщают только «угадала» (награда 1) или «не угадала» (награда 0). Правильный класс функции обучения не сообщают. По этой обратной связи программа постепенно меняет свои числа — веса.

Это **контекстный бандит**, простой частный случай обучения с подкреплением. «Контекст» — признаки текущего образца, «действие» — выбранный класс. Здесь нет игры, движения по состояниям или планирования будущих наград: выбор для одного вина не меняет следующий образец. Реальные данные Wine превращены в учебную среду; это не реальные логи действий.

**EN.** Imagine a program seeing a wine's chemical measurements and choosing class 0, 1 or 2. After choosing, it receives only “correct” (reward 1) or “incorrect” (reward 0). The training function is not told the correct class. It gradually changes its numbers, called weights, using this feedback.

This is a **contextual bandit**, a simple reinforcement-learning setting. The context is the current sample's features; the action is the chosen class. There is no game, state transition or planning of future rewards: choosing for one wine does not change the next sample. Real Wine data provide a teaching environment, not real action logs.

| Обозначение / Name | Простыми словами / Plain meaning | Форма здесь / Shape here |
|---|---|---|
| `x` | Признаки одного вина после обработки / Processed features of one wine | `(14,)`: 13 измерений + константа / measurements + constant |
| `contexts` | Несколько строк `x` / Several rows of `x` | `(N, 14)` |
| `W` | По одной строке весов для каждого действия / One weight row per action | `(3, 14)` |
| `scores = W @ x` | Три оценки ожидаемой награды, пока не вероятности / Three reward estimates, not probabilities | `(3,)` |
| `action` | Номер выбранного класса / Chosen class index | `0`, `1` или / or `2` |
| `reward` | Угадали или нет / Whether the guess was correct | `1.0` или / or `0.0` |
| `epsilon` | Как часто пробуем случайное действие / How often to try a random action | число от 0 до 1 / number from 0 to 1 |
| `alpha` | Размер изменения весов / Weight-update step size | здесь / here `0.2` |
| `epoch` | Один проход по всем Train-объектам / One pass over all training samples | целое число / integer |
| `seed` | Начальное значение генератора случайности для повторяемости / Random-generator seed for reproducibility | целое число / integer |

**План / Roadmap:** `epsilon_probs` превращает оценки в вероятности выбора → `update_action` меняет одну строку весов → `train_bandit` соединяет эти шаги → `evaluate_policy` проверяет готовые веса. / Convert scores to action probabilities → update one weight row → combine these steps into training → evaluate the learned weights.

**Как работать / How to work:** выполните подготовку данных, затем по одному из четырёх блоков. В каркасах заменяйте `...`; если есть `raise NotImplementedError`, удалите его после заполнения функции. `NotImplementedError` означает незавершённое задание, а `assert` проверяет результат. При ошибке сначала проверьте указанный блок. Готовые проверки и эксперимент переписывать не нужно. / Run data preparation, then complete one of the four blocks at a time. Replace `...` in the skeletons and remove `raise NotImplementedError` after completing each function. That exception means the exercise is unfinished; `assert` checks its result. Start debugging the indicated block. You do not need to rewrite the supplied checks or experiment.

**Подготовка данных уже написана / Data preparation is provided:** 106 Train, 36 Validation, 36 Test. `StandardScaler` учит среднее и масштаб только на Train. `encode` применяет эти числа, добавляет постоянный признак 1 (для свободного члена модели), затем делит каждую строку на её длину. Это помогает держать размеры обновлений сопоставимыми. Test открываем только после выбора epsilon. / The scaler learns means and scales on Train only. `encode` applies them, adds a constant feature 1 for an intercept, and divides each row by its length to keep update sizes comparable. Open Test only after selecting epsilon.

In [ ]:
import json, platform, time # Журнал, версия Python и часы / Log, Python version and clock.
from pathlib import Path # Путь к журналу / Experiment log path.
import numpy as np # Векторы контекстов и весов / Context and weight vectors.
import pandas as pd # Таблица запусков / Run comparison table.
import matplotlib.pyplot as plt # Графики и матрица ошибок / Plots and confusion matrix.
import sklearn # Зафиксировать версию библиотеки / Record the library version.
from sklearn.datasets import load_digits, load_wine # Встроенные реальные данные / Bundled real datasets.
from sklearn.model_selection import train_test_split # Стратифицированное разбиение / Stratified splitting.
from sklearn.preprocessing import StandardScaler, normalize # Train-scaler и нормировка объектов / Train scaler and row normalisation.
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay # Независимые метрики оценки / Independent evaluation metrics.
STUDENT_NAME = '' # Запишите ФИО / Enter your full name.
STUDENT_GROUP = '' # Запишите группу / Enter your group.
data = load_wine() # Загрузить исходные наблюдения / Load original observations.
X, y = data.data.astype(float), data.target.astype(int) # Признаки до решения и скрытая от learner метка / Pre-action features and learner-hidden label.
all_ids = np.arange(len(y)) # ID исходных строк для аудита / Original row IDs for auditing.
train_ids, rest_ids = train_test_split(all_ids, test_size=0.4, stratify=y, random_state=2026) # Отделить Train до обучения / Separate Train before fitting.
val_ids, test_ids = train_test_split(rest_ids, test_size=0.5, stratify=y[rest_ids], random_state=2026) # Разделить остаток на Validation и Test / Split remainder into Validation and Test.
assert not (set(train_ids)&set(val_ids) or set(train_ids)&set(test_ids) or set(val_ids)&set(test_ids)) # Проверить непересечение / Verify disjoint splits.
scaler = StandardScaler().fit(X[train_ids]) # Оценить параметры только на Train / Fit statistics on Train only.
def encode(ids): # Преобразование без повторного fit / Transform without refitting.
    z = scaler.transform(X[ids]) # Применить фиксированный scaler / Apply the fixed scaler.
    return normalize(np.column_stack([z, np.ones(len(z))])) # Добавить bias и нормировать строку / Append bias and normalise each row.
X_train, X_val = encode(train_ids), encode(val_ids) # Подготовить разрешённые части / Prepare permitted splits.
y_train, y_val = y[train_ids], y[val_ids] # Метки доступны среде и оценщику / Labels belong to environment and evaluator.
K = len(data.target_names) # Число допустимых действий из данных / Number of actions from data.
print('train / validation / test:', len(train_ids), len(val_ids), len(test_ids), 'actions:', K) # Показать размеры протокола / Display protocol sizes.
print('features:', data.feature_names) # Показать смысл входных измерений / Display input feature names.


## Блок 1. Как выбрать действие? / Block 1. How do we choose an action?

**RU. Зачем функция:** если всегда брать максимальную оценку, мы можем так и не попробовать другие действия. Поэтому иногда исследуем: выбираем случайно. Политика — просто правило выбора действия. Здесь с вероятностью `1-epsilon` выбираем лучший score, а с вероятностью `epsilon` выбираем равномерно среди **всех** действий, включая лучшее.

При трёх действиях и epsilon=0.3 каждому достаётся 0.3/3=0.1. Лучшему дополнительно 0.7: его итоговая вероятность 0.8. Это вероятность **выбора действия**, а не вероятность правильного класса. При равных scores берём первый максимум (`np.argmax`). При epsilon=0 всегда берём максимум, при epsilon=1 все действия равновероятны.

**EN. Why this function:** always taking the largest score could prevent us from ever trying alternatives. A policy is simply a rule for choosing an action. With probability `1-epsilon` take the best score; with probability `epsilon` choose uniformly among **all** actions, including the best.

For three actions and epsilon=0.3, each gets 0.1; the best gets an additional 0.7, so its total probability is 0.8. This is an **action-selection** probability, not a class probability. Use the first maximum on ties (`np.argmax`). Epsilon=0 always takes the maximum; epsilon=1 gives a uniform choice.

### Пример на реальных частотах Train / Example using actual training frequencies
Частоты ниже нужны только для знакомства с `scores` и `argmax`; это ещё не обученная модель. / The frequencies below only illustrate scores and argmax; this is not a learned model.

In [ ]:
counts = np.bincount(y_train, minlength=K) # Посчитать реальные классы только на Train / Count actual training classes.
example_scores = counts / counts.sum() # Частоты служат примером оценок / Frequencies provide example scores.
print('scores:', example_scores, 'greedy:', int(np.argmax(example_scores))) # Показать первый максимум / Display the first maximum.


### Задание 1 / Exercise 1

**RU.** Напишите epsilon_probs(scores, epsilon). Вход: непустой конечный одномерный вектор, ε∈[0,1]. Верните NumPy-вектор той же длины; каждому действию ε/K, первому максимальному score дополнительно 1−ε. При неверном входе выбросьте ValueError. Не меняйте scores. Проверьте ε=0, ε=1 и равенство scores; объясните, почему исследование иногда выбирает лучшее действие.

**EN.** Write epsilon_probs(scores, epsilon). Input: a finite nonempty 1D vector and epsilon in [0,1]. Return an equally sized NumPy vector: epsilon/K for every action, plus 1-epsilon for the first maximum. Raise ValueError for invalid inputs. Do not mutate scores. Check epsilon=0, epsilon=1 and ties; explain why exploration can also select the best action.

**По шагам / Step by step:**
1. Преобразуйте scores в числовой NumPy-массив. Проверка входа уже дана в каркасе. / Convert scores to a numeric array; input validation is provided.
2. Найдите число действий `K` и индекс первого максимума (`np.argmax`). / Find the action count and first maximum index.
3. Создайте новый массив из K одинаковых чисел `epsilon/K` через `np.full`. / Create K equal probabilities with `np.full`.
4. Прибавьте `1-epsilon` к одному элементу — лучшему действию. Верните массив. / Add `1-epsilon` to the best action's entry and return the array.

**Результат / Output:** `(3,)`, числа неотрицательны, сумма 1. Функция пока ничего не выбирает: только готовит вероятности. / Nonnegative entries summing to one; the function only prepares probabilities, it does not sample an action.

In [ ]:
def epsilon_probs(scores, epsilon): # Вероятности выбора / Action probabilities.
    scores = np.array(scores, dtype=float, copy=True) # Новая числовая копия / New numeric copy.
    if scores.ndim != 1 or scores.size == 0 or not np.isfinite(scores).all(): # Проверить вектор / Validate the vector.
        raise ValueError('scores must be a nonempty finite 1D vector') # Объяснить неверный вход / Explain invalid input.
    if np.ndim(epsilon) != 0 or not np.isfinite(epsilon) or not 0 <= epsilon <= 1: # Проверить epsilon / Validate epsilon.
        raise ValueError('epsilon must be a finite scalar in [0, 1]') # Объяснить диапазон / Explain valid range.
    n_actions = len(scores) # Сколько действий / Number of actions.
    best = ... # TODO: индекс первого максимума / Index of the first maximum.
    probabilities = ... # TODO: n_actions одинаковых вероятностей / Equal exploration probabilities.
    # TODO: прибавьте 1-epsilon к probabilities[best] / Add 1-epsilon to probabilities[best].
    raise NotImplementedError('Complete Exercise 1, then remove this line') # Удалите после заполнения / Remove when completed.
    return probabilities # Вернуть распределение / Return the distribution.

### Проверка 1 / Check 1

In [ ]:
p = epsilon_probs(example_scores, 0.2) # Проверить распределение на реальных частотах / Check probabilities on actual frequencies.
expected = np.full(K, 0.2 / K) # Независимо вычислить исследовательскую массу / Independently calculate exploration mass.
expected[np.argmax(example_scores)] += 0.8 # Добавить использование по контракту / Add exploitation according to the contract.
assert np.allclose(p, expected) and np.isclose(p.sum(), 1) and (p >= 0).all() # Проверить все вероятности / Verify all probabilities.
assert np.allclose(epsilon_probs(example_scores, 1), np.full(K, 1 / K)) # Проверить полное исследование / Check pure exploration.
assert epsilon_probs(np.zeros(K), 0)[0] == 1 # Проверить первый индекс при равенстве / Check first-index tie breaking.
for bad in [-0.01, 1.01]: # Проверить запрещённые значения epsilon / Check invalid epsilon values.
    try: # Ожидать отказ при неверном аргументе / Expect rejection of invalid input.
        epsilon_probs(example_scores, bad) # Передать неверный диапазон / Pass an out-of-range value.
    except ValueError: # Принять корректный тип ошибки / Accept the specified exception.
        continue # Перейти к следующему случаю / Continue to the next case.
    raise AssertionError('epsilon must be validated') # Не пропускать молчаливую ошибку / Reject silent invalid behaviour.
print('T1 passed') # Подтвердить контракт распределения / Confirm the probability contract.


## Блок 2. Как учиться на одном ответе? / Block 2. How do we learn from one response?

**RU. Зачем функция:** после действия нужно немного поправить модель. `W[action]` — строка весов выбранного действия; `W[action] @ x` — сумма попарных произведений весов и признаков. Она даёт текущую оценку награды `q`. Посчитаем ошибку `reward-q`. Если ожидали 0.4, а получили 1, ошибка +0.6: оценку следует поднять. Если получили 0, ошибка −0.4: снизить.

Правило изменения: `alpha * (reward-q) * x`. Прибавляем этот вектор только к строке выбранного действия. За остальные действия награды не получили, поэтому их строки не меняем. Возвращаем **копию**: `W_new = W` не копирует массив, нужна `W.copy()`.

**EN. Why this function:** after acting, adjust the model slightly. `W[action]` is the chosen action's weight row; `W[action] @ x` sums pairwise weight-feature products to estimate reward `q`. The prediction error is `reward-q`. Expecting 0.4 and receiving 1 gives +0.6, so raise the estimate; receiving 0 gives −0.4, so lower it.

Add the vector `alpha * (reward-q) * x` only to the chosen row. Other actions produced no observed rewards, so leave their rows unchanged. Return a **copy**: `W_new = W` does not copy an array; use `W.copy()`.

### Пример: признаки и награда одного настоящего вина / Example: one real wine's features and reward

In [ ]:
example_x = X_train[0] # Взять контекст реального объекта / Take a real observation context.
example_action = 0 # Зафиксировать демонстрационное действие / Fix the example action.
example_reward = float(example_action == y_train[0]) # Среда возвращает награду только выбранного действия / Environment returns only chosen-action reward.
print('norm:', np.linalg.norm(example_x), 'action:', example_action, 'reward:', example_reward) # Показать вход одного обновления / Display one update's input.
demo_weights = 0.4 * example_x # Веса только для показа скалярного произведения / Weights only to illustrate the dot product.
demo_score = demo_weights @ example_x # При длине x=1 получится 0.4 / With unit-length x this gives 0.4.
print('estimate:', demo_score, 'error if reward=1:', 1 - demo_score, 'error if reward=0:', 0 - demo_score) # Сравните знаки ошибок / Compare error signs.


### Задание 2 / Exercise 2

**RU.** Напишите update_action(W, x, action, reward, alpha). Для корректных числовых входов верните новую матрицу W с единственным изменением: строка action += alpha*(reward−W[action]@x)*x. Исходная W неизменна; x уже нормирован. Не используйте истинный класс и не меняйте другие строки. Покажите обновление при наградах 1 и 0 на реальном x; объясните знак ошибки.

**EN.** Write update_action(W, x, action, reward, alpha). For valid numeric inputs return a new matrix with only row action changed by alpha*(reward-W[action]@x)*x. Preserve the input W; x is already normalised. Do not access the true class or update other rows. Demonstrate rewards 1 and 0 on a real x and explain the error sign.

**По шагам / Step by step:**
1. Создайте независимую копию `W`. / Make an independent copy of W.
2. Посчитайте текущую оценку выбранного действия: скалярное произведение его строки с `x`. / Compute the chosen row's dot product with x.
3. Из полученной награды вычтите оценку. / Subtract the estimate from the observed reward.
4. Прибавьте к выбранной строке копии `alpha * error * x`. / Add alpha times error times x to the chosen row of the copy.
5. Верните копию. / Return the copy.

**Результат / Output:** матрица `(3, 14)`; меняется только одна строка. / A matrix of the same shape, with only one row changed.

In [ ]:
def update_action(W, x, action, reward, alpha): # Одно обновление / One update.
    updated = W.copy() # Не менять исходные веса / Preserve the input weights.
    estimate = ... # TODO: скалярное произведение выбранной строки и x / Dot product of chosen row and x.
    error = ... # TODO: наблюдаемая награда минус оценка / Observed reward minus estimate.
    # TODO: измените только updated[action] / Update only updated[action].
    raise NotImplementedError('Complete Exercise 2, then remove this line') # Удалите после заполнения / Remove when completed.
    return updated # Вернуть новые веса / Return new weights.

### Проверка 2 / Check 2

In [ ]:
W0 = np.zeros((K, X_train.shape[1])) # Подготовить нулевые веса для реального контекста / Prepare zero weights for a real context.
a = int(y_train[0]) # Выбрать действие с известной среде наградой один / Choose an action with environment-known reward one.
W1 = update_action(W0, X_train[0], a, 1.0, 0.2) # Выполнить один шаг / Perform one update.
reference = W0.copy() # Независимо построить ожидаемый результат / Independently form the expected result.
reference[a] = 0.2 * X_train[0] # При нулевом прогнозе ошибка равна единице / With zero prediction the error is one.
assert np.allclose(W1, reference) and np.count_nonzero(W0) == 0 # Проверить результат и отсутствие мутации / Verify result and no input mutation.
W2 = update_action(W1, X_train[0], a, 0.0, 0.2) # Проверить отрицательную ошибку прогноза / Check a negative prediction error.
reference2 = W1.copy() # Сохранить контрольную копию / Keep a reference copy.
reference2[a] -= 0.2 * float(W1[a] @ X_train[0]) * X_train[0] # Независимо вычислить корректировку / Independently calculate the correction.
assert np.allclose(W2, reference2) and not np.shares_memory(W2, W1) # Проверить значения и независимость копии / Check values and independent storage.
print('T2 passed') # Подтвердить обновление одной строки / Confirm the selected-row update.


## Блок 3. Как соединить шаги в обучение? / Block 3. How do we combine the steps?

**RU. Зачем функция:** повторить выбор и обновление для всех Train-объектов. Одна эпоха — один проход; на каждой эпохе перемешиваем порядок. `feedback` — обычная функция, переданная как аргумент (её также называют callback). Вызов `feedback(index, action)` возвращает 0 или 1. Только она смотрит правильную метку. `index` — позиция строки в переданном `contexts`, не исходный ID Wine.

Для одного объекта: взять `x` → получить `W @ x` → вызвать функцию 1 → выбрать действие через `rng.choice(..., p=...)` → один раз запросить награду → записать событие → вызвать функцию 2. В начале веса нулевые. Seed делает случайные выборы повторяемыми. Не сравнивайте действие с `y_train` внутри обучения: это работа среды.

**EN. Why this function:** repeat selection and updating for every training sample. One epoch is one pass; shuffle the order for each epoch. `feedback` is an ordinary function passed as an argument, also called a callback. Calling `feedback(index, action)` returns 0 or 1. Only that function accesses the true label. `index` is the row position in the supplied `contexts`, not the original Wine ID.

For one sample: take `x` → compute `W @ x` → call function 1 → sample with `rng.choice(..., p=...)` → request one reward → log the event → call function 2. Start with zero weights. A seed makes random choices repeatable. Do not compare with `y_train` inside the learner; the environment does that.

**Журнал / Log:** одна строка `(epoch, index, action, reward, probability)` на действие. Последнее число — вероятность выбранного действия **до** обучения на его награде. Это позволяет восстановить, как принималось решение. / One row per action; the last number is its selection probability **before** learning from its reward, allowing us to reconstruct the decision.

### Пример функции среды / Example environment function

In [ ]:
def train_feedback(index, action): # Среда скрывает метку от функции обновления / Environment hides the label from the update function.
    return float(action == y_train[index]) # Возвращать лишь награду за выбранное действие / Return only reward for the selected action.
print('first action reward:', train_feedback(0, 0)) # Продемонстрировать один запрос среды / Demonstrate one environment query.


### Задание 3 / Exercise 3

**RU.** Напишите train_bandit(contexts, feedback, n_actions, epsilon, alpha, epochs, seed). Все аргументы корректны. Создайте локальный default_rng(seed), нулевую W формы (n_actions, n_features). На каждой эпохе используйте rng.permutation, затем epsilon_probs(W@x), rng.choice и единственный feedback(index, action). Запишите (epoch,index,action,reward,p[action]) ДО update_action. Верните W и float-массив истории (epochs*n_rows,5). В функции не обращайтесь к y_train/y/Validation/Test; только callback имеет метки. Повтор seed обязан повторить результат.

**EN.** Write train_bandit(contexts, feedback, n_actions, epsilon, alpha, epochs, seed), assuming valid arguments. Use local default_rng(seed) and zero weights shaped (n_actions,n_features). Each epoch uses rng.permutation; each step uses epsilon_probs(W@x), rng.choice, and exactly one feedback(index,action). Log (epoch,index,action,reward,p[action]) BEFORE update_action. Return W and a float history array (epochs*n_rows,5). Never access y_train/y/Validation/Test inside the learner; only the callback has labels. Repeating a seed must reproduce the result.

**По шагам / Step by step:** цикл и журнал уже даны. Заполните пять пропусков внутри цикла: / The loops and log are supplied. Fill the five blanks inside the loop:
1. `scores`: оценки всех действий на текущем x. / Scores for all actions on the current x.
2. `p`: вызов вашей функции 1 с scores и epsilon. / Call your function 1 with scores and epsilon.
3. `action`: случайный выбор из n_actions с вероятностями p; используйте `rng.choice`. / Sample from n_actions using probabilities p with rng.choice.
4. `reward`: один вызов переданной feedback. / One call to the supplied feedback function.
5. `W`: результат вашей функции 2 с текущими весами, x, действием, наградой и alpha. / Result of your function 2 using current weights, x, action, reward and alpha.

**Результат / Output:** пара `(W, history)`. При 12 строках и 2 эпохах history имеет 24 строки и 5 столбцов. / A pair; with 12 samples and 2 epochs the history has 24 rows and 5 columns.

In [ ]:
def train_bandit(contexts, feedback, n_actions, epsilon, alpha, epochs, seed): # Обучение на Train / Training on Train.
    rng = np.random.default_rng(seed) # Локальная повторяемая случайность / Local reproducible randomness.
    W = np.zeros((n_actions, contexts.shape[1])) # Начать с нулевых оценок / Start with zero estimates.
    history = [] # Список событий / Event list.
    for epoch in range(epochs): # Несколько проходов / Multiple passes.
        for index in rng.permutation(len(contexts)): # Все строки в новом порядке / All rows in a new order.
            x = contexts[index] # Признаки текущего вина / Current wine's features.
            scores = ... # TODO 1: оценки W на x / Scores from W on x.
            p = ... # TODO 2: вызов epsilon_probs / Call epsilon_probs.
            action = ... # TODO 3: выбор через rng.choice / Sample with rng.choice.
            reward = ... # TODO 4: один вызов feedback / One call to feedback.
            history.append((epoch, index, action, reward, p[action])) # Сохранить вероятность до обновления / Log pre-update probability.
            W = ... # TODO 5: вызов update_action / Call update_action.
    return W, np.asarray(history, dtype=float).reshape(-1, 5) # Веса и таблица событий / Weights and event table.

### Проверка 3 / Check 3

In [ ]:
calls = [] # Записать обращения к среде / Record environment calls.
def checked_feedback(index, action): # Контролировать допустимые запросы / Validate environment queries.
    assert 0 <= index < 12 and 0 <= action < K # Не допустить чужих объектов или действий / Reject invalid observations or actions.
    calls.append((index, action)) # Сохранить порядок запросов / Save query order.
    return float(action == y_train[index]) # Вернуть награду реального наблюдения / Return reward from the real observation.
W_check, log_check = train_bandit(X_train[:12], checked_feedback, K, 0.2, 0.2, 2, 73) # Выполнить короткий реальный прогон / Run a short real-data trial.
assert W_check.shape == (K, X_train.shape[1]) and log_check.shape == (24, 5) # Проверить формы выходов / Verify output shapes.
assert len(calls) == 24 and np.array_equal(log_check[:, 1:3], np.asarray(calls)) # Сопоставить журнал с фактическими вызовами / Match logs to actual calls.
replay = np.zeros_like(W_check) # Восстановить веса независимо из событий / Independently reconstruct weights from events.
for epoch, index, action, reward, propensity in log_check: # Читать события в исходном порядке / Read events in original order.
    index, action = int(index), int(action) # Восстановить целочисленные индексы / Recover integer indices.
    assert reward == float(action == y_train[index]) # Проверить награду по исходным данным / Check reward against source labels.
    scores = replay @ X_train[index] # Получить оценки до обновления / Compute pre-update estimates.
    probability = 0.2 / K + (0.8 if action == int(np.argmax(scores)) else 0.0) # Независимо проверить вероятность / Independently check propensity.
    assert np.isclose(propensity, probability) # Обнаружить логирование после обновления / Detect post-update logging.
    replay[action] += 0.2 * (reward - scores[action]) * X_train[index] # Восстановить только выбранную строку / Replay only the chosen row.
assert np.allclose(W_check, replay) # Сверить финальные веса с журналом / Match final weights to the log.
for epoch in range(2): # Проверить каждый проход / Check each epoch.
    assert sorted(log_check[log_check[:, 0] == epoch, 1]) == list(range(12)) # Каждый объект посещается ровно один раз / Visit each observation exactly once.
W_again, log_again = train_bandit(X_train[:12], lambda i, a: float(a == y_train[i]), K, 0.2, 0.2, 2, 73) # Повторить тот же seed / Repeat the same seed.
assert np.array_equal(log_check, log_again) and np.allclose(W_check, W_again) # Проверить воспроизводимость / Verify reproducibility.
print('T3 passed') # Подтвердить контракт обучения / Confirm the training contract.


## Блок 4. Как проверить готовую модель? / Block 4. How do we evaluate the learned model?

**RU. Зачем функция:** узнать, чему модель научилась, на отдельных Validation-объектах. Теперь веса не меняем и случайных проб не делаем. Для всех объектов сразу `contexts @ W.T` даёт таблицу `(N, K)`: строки — вина, столбцы — действия. `.T` поворачивает матрицу весов для умножения. `np.argmax(scores, axis=1)` берёт лучший столбец в каждой строке.

Сравним выбранные действия с метками. Средняя награда — доля совпадений, то есть accuracy. Macro-F1 — среднее F1 по трём классам; матрица ошибок показывает, какие классы перепутаны (строка — истина, столбец — прогноз). Здесь можно использовать готовые функции sklearn. Метки разрешены оценщику, но не функции обучения. Эти scores не обязаны складываться в 1: это линейные оценки награды.

**EN. Why this function:** measure what the model learned on separate validation samples. Keep weights fixed and make no random exploratory choices. `contexts @ W.T` produces an `(N, K)` table: wine samples in rows, actions in columns. `.T` transposes the weight matrix for multiplication. `np.argmax(scores, axis=1)` selects the best column in each row.

Compare chosen actions with labels. Mean reward is the fraction of matches, equal to accuracy. Macro-F1 averages F1 across the three classes; the confusion matrix records true classes in rows and predictions in columns. You may use sklearn metric functions. The evaluator may access labels; the training function may not. Scores need not sum to one: they are linear reward estimates.

### Пример: всегда выбирать самый частый Train-класс / Example: always choose the most common training class

In [ ]:
majority = int(np.argmax(np.bincount(y_train, minlength=K))) # Выбрать константу по Train / Select the constant action using Train.
majority_val = np.full(len(y_val), majority) # Предсказать константу на Validation / Predict the constant on Validation.
print('majority validation accuracy:', accuracy_score(y_val, majority_val)) # Показать реальный простой baseline / Display a real simple baseline.


### Задание 4 / Exercise 4

**RU.** Напишите evaluate_policy(W, contexts, labels). Верните dict mean_reward, accuracy, macro_f1, matrix, predicted. Действие — первый argmax(contexts@W.T) по строке; matrix формы (K,K), строки=true, столбцы=predicted; macro-F1 по всем 0..K−1, zero_division=0. Метки доступны только оценщику. Не изменяйте W и не обучайте scaler. Затем выполните фиксированное сравнение ниже, запишите выбор до Test, визуализируйте ошибки и ответьте на четыре вопроса отчёта.

**EN.** Write evaluate_policy(W, contexts, labels). Return a dict with mean_reward, accuracy, macro_f1, matrix, predicted. Use row-wise first argmax(contexts@W.T); matrix is (K,K), rows=true and columns=predicted; macro-F1 covers 0..K-1 with zero_division=0. Labels are available only to the evaluator. Never mutate W or fit the scaler. Then run the fixed comparison below, record selection before Test, visualise errors, and answer the four report questions.

**По шагам / Step by step:**
1. Получите scores для всех строк: матричное умножение contexts на транспонированную W. / Multiply contexts by transposed W for all-row scores.
2. Найдите лучший индекс по каждой строке через argmax с axis=1. / Take row-wise argmax with axis=1.
3. Получите награды: сравните predicted с labels, затем усредните. / Compare predictions with labels and average the rewards.
4. Заполните словарь: `accuracy_score`, `f1_score` и `confusion_matrix`. Для последних двух передайте `labels=np.arange(K)`, чтобы не потерять классы. У F1 нужны `average='macro', zero_division=0`. / Fill the dictionary with sklearn metrics; specify all K labels, macro averaging and zero_division=0 for F1.

**Результат / Output:** словарь из пяти полей; `predicted` формы `(N,)`, `matrix` формы `(3,3)`, остальные поля — числа. / Five fields: prediction vector, confusion matrix, and three scalar metrics.

In [ ]:
def evaluate_policy(W, contexts, labels): # Только оценка, без обучения / Evaluation only, no training.
    n_actions = W.shape[0] # Число классов / Number of classes.
    scores = ... # TODO: оценки для всех объектов / Scores for all samples.
    predicted = ... # TODO: максимум в каждой строке / Maximum in each row.
    mean_reward = ... # TODO: средняя доля совпадений / Mean fraction of matches.
    raise NotImplementedError('Complete Exercise 4, then remove this line') # Удалите после заполнения / Remove when completed.
    return { # Сохраните имена полей / Preserve the field names.
        'mean_reward': mean_reward, # Средняя награда / Mean reward.
        'accuracy': ..., # TODO: accuracy_score(labels, predicted) / Accuracy metric.
        'macro_f1': ..., # TODO: F1 по всем классам / F1 across all classes.
        'matrix': ..., # TODO: confusion_matrix, строки true / Confusion matrix, true rows.
        'predicted': predicted, # Выбранные действия / Selected actions.
    }

### Проверка 4 / Check 4

In [ ]:
before = W_check.copy() # Сохранить веса до оценки / Save weights before evaluation.
result = evaluate_policy(W_check, X_val, y_val) # Оценить только Validation / Evaluate Validation only.
pred = np.argmax(X_val @ W_check.T, axis=1) # Независимо получить жадные решения / Independently obtain greedy actions.
cm = np.zeros((K, K), dtype=int) # Подготовить контрольную матрицу / Prepare a reference matrix.
np.add.at(cm, (y_val, pred), 1) # Посчитать пары true/predicted без confusion_matrix / Count true/predicted pairs independently.
assert np.array_equal(result['matrix'], cm) and cm.sum() == len(y_val) # Проверить ориентацию и число объектов / Verify orientation and observation count.
assert np.array_equal(result['predicted'], pred) and np.array_equal(before, W_check) # Проверить предсказания и заморозку / Verify predictions and frozen weights.
assert np.isclose(result['mean_reward'], np.mean(pred == y_val)) and np.isclose(result['accuracy'], result['mean_reward']) # Проверить награду и accuracy / Check reward and accuracy.
assert np.isclose(result['macro_f1'], f1_score(y_val, pred, labels=np.arange(K), average='macro', zero_division=0)) # Проверить учёт всех классов / Verify all-class macro-F1.
print('T4 passed') # Подтвердить честную оценку / Confirm valid evaluation.


## Перед сравнением / Before comparison

Запишите 2–3 предложения: как epsilon=0.3 относительно 0.1 повлияет на train reward и на замороженную validation-награду? Меняем только epsilon. Не смотрите Test. / Write 2–3 sentences predicting how epsilon=0.3 versus 0.1 affects training reward and frozen validation reward. Change only epsilon. Do not inspect Test.

**Мой прогноз / My prediction:**

## Готовый эксперимент A: сравнение на Validation / Supplied experiment A: validation comparison

Заполните прогноз выше, затем запустите ячейку. Два epsilon × три seed = шесть запусков. Сравниваем среднюю Validation-награду; при равенстве оставляем 0.1. Ничего дописывать не нужно. / Complete the prediction above, then run the cell. Two epsilon values times three seeds give six runs. Select by mean validation reward; ties retain 0.1. No additional code required.

In [ ]:
SEEDS = [2026, 2027, 2028] # Зафиксировать запуски до сравнения / Prespecify seeds before comparison.
EPSILONS = [0.1, 0.3] # Менять только исследование; 0.1 — baseline / Vary only exploration; 0.1 is the baseline.
ALPHA, EPOCHS = 0.2, 60 # Сохранить одинаковые шаг и бюджет / Keep learning rate and budget fixed.
models, histories, rows = {}, {}, [] # Собрать все запуски, не выбирать удачный seed / Retain all runs rather than selecting a lucky seed.
for epsilon in EPSILONS: # Сравнить два заранее заданных варианта / Compare the two prespecified variants.
    for seed in SEEDS: # Использовать одинаковый набор seed / Use the same seed set.
        W, history = train_bandit(X_train, train_feedback, K, epsilon, ALPHA, EPOCHS, seed) # Обучать только на Train / Train only on training data.
        models[epsilon, seed], histories[epsilon, seed] = W, history # Сохранить веса и журнал / Save weights and log.
        report = evaluate_policy(W, X_val, y_val) # Оценить замороженную политику на Validation / Evaluate the frozen policy on Validation.
        rows.append(dict(epsilon=epsilon, seed=seed, mean_reward=report['mean_reward'], macro_f1=report['macro_f1'], train_reward=float(history[:, 3].mean()))) # Не смешивать Train и Validation reward / Keep training and validation rewards distinct.
validation_table = pd.DataFrame(rows) # Свести все шесть запусков / Summarise all six runs.
display(validation_table) # Показать индивидуальные результаты / Display individual results.
means = validation_table.groupby('epsilon')['mean_reward'].mean() # Сравнить среднюю Validation-награду / Compare mean validation reward.
selected_epsilon = 0.3 if means.loc[0.3] > means.loc[0.1] else 0.1 # При точном равенстве оставить baseline / Keep baseline on an exact tie.
selection = dict(epsilon=selected_epsilon, seeds=SEEDS, alpha=ALPHA, epochs=EPOCHS, criterion='mean validation reward; exact tie -> 0.1') # Зафиксировать протокол / Freeze the protocol.
Path('selection_rl.json').write_text(json.dumps(selection, indent=2)) # Сохранить выбор перед Test / Persist selection before Test.


## Готовый эксперимент B: разбор ошибки / Supplied experiment B: inspect an error

Веса уже обучены. Здесь показаны график награды на Train, матрица Validation и первая ошибка с признаками. Запишите факты для ответа 3. Причина ошибки — гипотеза, если её отдельно не проверяли. / Weights are already trained. Inspect training rewards, the validation confusion matrix and the first failure with its features. Record facts for answer 3; a proposed cause is a hypothesis unless independently tested.

In [ ]:
for epsilon in EPSILONS: # Показать динамику всех запусков каждого варианта / Plot all runs of each variant.
    curves = [histories[epsilon, seed][:, 3].reshape(EPOCHS, len(X_train)).mean(axis=1) for seed in SEEDS] # Усреднить награды внутри эпох / Average training rewards within epochs.
    plt.plot(np.arange(1, EPOCHS+1), np.mean(curves, axis=0), label=f'epsilon={epsilon}') # Показать среднюю траекторию / Plot the mean trajectory.
plt.xlabel('Training epoch'); plt.ylabel('Mean training reward'); plt.legend(); plt.show() # Подписать тренировочный график / Label the training plot.
chosen_val = evaluate_policy(models[selected_epsilon, SEEDS[0]], X_val, y_val) # Выбрать заранее фиксированный seed для иллюстрации / Use the prespecified first seed for illustration.
ConfusionMatrixDisplay(chosen_val['matrix'], display_labels=data.target_names).plot(cmap='Blues') # Показать Validation-матрицу / Display the validation matrix.
plt.title('Validation; first prespecified seed'); plt.show() # Не подменять иллюстрацию Test / Label validation explicitly.
wrong = np.flatnonzero(chosen_val['predicted'] != y_val) # Найти реальные Validation-ошибки / Find actual validation errors.
print('Validation errors:', len(wrong), 'of', len(y_val)) # Сообщить также случай без ошибок / Also report the zero-error case.
if len(wrong): # Разобрать первую ошибку, если она есть / Inspect the first error when available.
    j = int(wrong[0]) # Зафиксировать локальный индекс / Fix the local index.
    print('source ID:', int(val_ids[j]), 'true:', int(y_val[j]), 'action:', int(chosen_val['predicted'][j]), 'scores:', models[selected_epsilon, SEEDS[0]] @ X_val[j]) # Показать проверяемые факты / Display auditable facts.
    display(pd.DataFrame([X[val_ids[j]]], columns=data.feature_names)) # Показать реальные входные признаки / Display actual input measurements.


## Готовый эксперимент C: финальная проверка / Supplied experiment C: final evaluation

Выбор сохранён в selection_rl.json. Теперь запускаем Test для каждого из трёх заранее заданных seed и сохраняем rl_experiment.json. После просмотра Test не меняйте epsilon и не выбирайте лучший seed. / Selection is saved in selection_rl.json. Evaluate all three prespecified seeds on Test and save rl_experiment.json. Do not change epsilon or select the best seed after viewing Test.

In [ ]:
X_test, y_test = encode(test_ids), y[test_ids] # Открыть Test только после фиксации выбора / Access Test only after freezing selection.
test_rows = [] # Сохранить все три финальных запуска / Retain all three final runs.
for seed in SEEDS: # Не выбирать лучший Test-seed / Never select the best test seed.
    result = evaluate_policy(models[selected_epsilon, seed], X_test, y_test) # Однократно оценить каждый заранее выбранный запуск / Evaluate each prespecified run once.
    test_rows.append(dict(seed=seed, mean_reward=result['mean_reward'], macro_f1=result['macro_f1'])) # Сохранить сопоставимые метрики / Save comparable metrics.
display(pd.DataFrame(test_rows)) # Показать разброс финальных результатов / Display variation across final runs.
print('test mean reward:', np.mean([r['mean_reward'] for r in test_rows])) # Усреднить без выбора победителя / Average without picking a winner.
experiment_log = dict(dataset='Wine', source='https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html', selection=selection, validation=rows, test=test_rows, split_ids=dict(train=train_ids.tolist(), validation=val_ids.tolist(), test=test_ids.tolist()), versions=dict(python=platform.python_version(), numpy=np.__version__, sklearn=sklearn.__version__), reward='1[action == original label]', limitation='Teaching contextual bandit, not real logged interventions') # Собрать проверяемый протокол / Assemble an auditable protocol.
Path('rl_experiment.json').write_text(json.dumps(experiment_log, ensure_ascii=False, indent=2)) # Сохранить журнал сдачи / Save the submission log.


## Отчёт / Report
1. Укажите источник, объект, признаки, метку, размеры частей, доступность измерений и единицу разбиения. Почему это контекстный бандит, а не последовательное управление? / State source, observation, features, target, split sizes, feature availability and split unit. Why is this a contextual bandit rather than sequential control?
2. Приведите прогноз до сравнения, все шесть validation-результатов, средние и выбранный epsilon. Объясните отличие тренировочной награды от оценки замороженной политики. / Give your prior prediction, all six validation results, means and selected epsilon. Explain training reward versus frozen-policy evaluation.
3. Разберите одну реальную validation-ошибку: source ID, true, action, scores и признаки ; если ошибок нет, сообщите это. Отделите наблюдение от гипотезы. Почему scores не обязаны быть вероятностями? / Analyse an actual validation failure: source ID, true, action, scores and features , or explicitly report none. Separate observation and hypothesis. Why are scores not necessarily probabilities?
4. Приведите три Test-результата и среднее, два ограничения переноса, следующий эксперимент без подбора по текущему Test, источники и AI-помощь. Почему оценку нельзя прямо перенести на реальные логи действий? / Give all three test results and their mean, two transfer limitations, a follow-up without tuning on current Test, sources and AI disclosure. Why does this evaluation not directly apply to real action logs?

**Ваш ответ / Your answer:**


### Заполните по своим выводам / Fill in from your own outputs

1. **Данные / Data:** объект / observation: ___; 13 исходных признаков / original features: ___; метка / target: ___; источник / source: ___. Train/Val/Test: ___/___/___. Scaler обучен на / fitted on ___. Действие не меняет следующий объект, поэтому / The action does not change the next observation, so ___. Укажите, когда доступны измерения. / State when measurements become available.
2. **Сравнение / Comparison:** мой прогноз выше / my prediction above: ___. Перенесите шесть строк из validation_table и два средних из means. / Copy the six validation_table rows and two means. Выбран / selected epsilon: ___, потому что / because ___. На Train действия иногда случайные, а на Validation / Training actions are sometimes random, whereas on validation ___.
3. **Ошибка / Error:** source ID ___; true ___; action ___; три scores / three scores ___. Приведите 2–3 измерения из показанной строки. / Give 2–3 measurements from the displayed row. Наблюдение / observation: ___. Возможное объяснение, пока гипотеза / possible explanation, still a hypothesis: ___. Scores не вероятности, потому что / Scores are not probabilities because ___. Если ошибок нет, явно напишите это. / Explicitly state if there are no errors.
4. **Итог / Conclusion:** три Test-награды / three test rewards: ___; среднее / mean: ___. Ограничение размера или источника данных / size or source limitation: ___. Отличие учебной награды от реальных действий / teaching reward versus real actions: ___. Следующая проверяемая гипотеза на новых данных без подбора по текущему Test / next testable hypothesis on new data without tuning on current Test: ___. Источники и помощь ИИ / sources and AI assistance: ___.


## Сдача и оценка / Submission and grading

Сдайте выполненный HW10_RL.ipynb и rl_experiment.json в свою папку ДЗ; заполните ФИО и группу. Код 80: T1–T4 по 20; анализ 20: четыре ответа по 5. Самопроверка не является оценкой. Срок — в матрице группы. Штраф: raw × max(0,1−late/14 суток), затем ×0.9 при отсутствии P10; отметку практики ставит преподаватель. / Submit the executed HW10_RL.ipynb and rl_experiment.json in your homework folder; fill in name/group. Code: 80 (20 per task); analysis: 20 (5 per answer). Self-checks are not grades. Use the group deadline matrix. Penalty: raw × max(0,1−lateness/14 days), then ×0.9 if P10 is unmarked; only the instructor changes practice attendance.